# NB 06 · Persistence

**Series:** Learning LangGraph through literate notebooks, notebook 6 of 7.
**Central question:** how does a run pause for a human decision, and continue exactly where it stopped?

In NB 04, topic c ended with a limited answer that the graph wrote on its own. In a regulated setting,
an answer resting on a single source should not go out without a person signing it off. That takes
three things a graph has not had so far: a way to **pause**, a **memory** of everything the run has
done, and a way to **resume** with the person's decision. This notebook adds all three.

## Learning goals

After this notebook you can:

- explain what a checkpointer saves, when it saves it, and why every run needs a `thread_id`;
- pause a run with `interrupt()` and resume it with `Command(resume=...)`;
- route a person's decision, so that approval and rejection take different paths;
- keep a paused run alive across a restart by saving checkpoints in a local SQLite file;
- recognise three persistence mistakes whose errors appear far from their cause.

## Setup

The scripted model replays three recordings from `qwen3:8b`: the claims of NB 03, the search queries
of NB 04, and the claim of one constructed document introduced below. Checkpoint files go to a
temporary folder that the notebook removes at the end.

In [ ]:
import shutil
import sqlite3
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from research_assistant.assistant import collect_primary, collect_secondary, make_score, write_answer
from research_assistant.evidence import CANONICAL_QUESTIONS, EVIDENCE_THRESHOLD, assess
from research_assistant.loop import LOOP_CONTRACT, MAX_ROUNDS, limitation_text, make_refine_query
from research_assistant.models import get_chat_model, get_llm_mode
from research_assistant.persistence import CONSTRUCTED_REPORT, conflicting_figures
from research_assistant.state import ResearchState, check_contract

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langgraph-checkpoint-sqlite"):
    print(f"{package} {version(package)}")

mode = get_llm_mode()
if mode == "disabled":
    mode = "fake"  # this notebook needs a model; the recorded replies stand in
llm = get_chat_model(mode, fixture=("nb03_claims.yaml", "nb04_queries.yaml", "nb06_constructed.yaml"))
print(f"Model mode: {mode}")

QUESTIONS = CANONICAL_QUESTIONS
CHECKPOINT_FOLDER = Path(tempfile.mkdtemp(prefix="nb06_"))


def thread(name: str) -> dict:
    return {"configurable": {"thread_id": name}}

## Concept: checkpoints, a pause, a decision

A **checkpointer** saves the complete state after every step of a run, under a name for that run:
the **thread**. Every call that belongs to the same run passes the same `thread_id`. With a
checkpointer, a run no longer has to finish in one go.

**`interrupt(value)`** inside a node stops the run at that point and hands `value` to whoever started
it: here, a draft answer and the reason for the review. The state up to that moment sits in the
checkpoint. **`Command(resume=...)`**, sent later on the same thread, delivers the person's answer.
The paused node then runs again from its beginning, and this time `interrupt()` returns the answer
instead of stopping.

```text
... --> [score] --router--+-- answer -------> [write_answer] -------> END
                          +-- search_again -> [refine_query] -> (search again)
                          +-- review -------> [human_review] --+-- publish --> [publish_reviewed] --> END
                                               (pauses here)   +-- withhold -> [withhold_answer] ---> END
```

The review fires on exactly two triggers, fixed in ADR 0007:

| Trigger | Case in this notebook |
|---|---|
| the quality gate still fails after the last search round | topic c, from the corpus |
| two qualifying sources contradict each other on a fact | a **constructed** variant of topic a |

No corpus topic produces the second trigger: in every topic, at least one side of a contradiction is
weak, outdated or a copy. So the notebook constructs the case openly, as the decision record
planned, rather than leaving that branch untested.

## Experiment

### 1 Memory per thread

The simplest checkpointer keeps everything in memory. A graph compiled with it remembers each thread's
state after the run, and every step along the way.

In [ ]:
def build_answer_only_graph(checkpointer):
    builder = StateGraph(ResearchState)
    builder.add_node("collect_primary", collect_primary)
    builder.add_node("collect_secondary", collect_secondary)
    builder.add_node("score", make_score(llm))
    builder.add_node("write_answer", write_answer)
    builder.add_edge(START, "collect_primary")
    builder.add_edge(START, "collect_secondary")
    builder.add_edge(["collect_primary", "collect_secondary"], "score")
    builder.add_edge("score", "write_answer")
    builder.add_edge("write_answer", END)
    return builder.compile(checkpointer=checkpointer)


memory_graph = build_answer_only_graph(InMemorySaver())
memory_graph.invoke({"question": QUESTIONS["a"]}, thread("topic-a"))

saved = memory_graph.get_state(thread("topic-a"))
print("saved answer:", saved.values["draft_answer"][:60], "...")
print("steps saved: ", len(list(memory_graph.get_state_history(thread("topic-a")))))
print("other thread:", memory_graph.get_state(thread("topic-b")).values)

The finished run's state stays available under its thread, together with a checkpoint for every step.
Another thread starts empty: threads keep runs apart, so two reviews never mix.

### 2 The router and the review step

The router of NB 04 gains one label, `review`, for both triggers. A credible contradiction goes to
review at once: searching again cannot settle a disagreement between two credible sources.

`human_review` builds what the reviewer needs to decide (the trigger and the draft answer) and calls
`interrupt()`. Whatever the person sends back is checked: anything other than `approve` or `reject` is
refused, because a typo in a decision must not be read as one.

In [ ]:
DECISIONS = ("approve", "reject")


def review_trigger(state: ResearchState) -> str:
    if assess(state["documents"], state["claims"]).unresolved:
        return f"qualifying sources contradict each other: {conflicting_figures(state)}"
    return f"the quality gate is not passed after {state.get('retrieval_round', 1)} of {MAX_ROUNDS} search rounds"


def route_with_review(state: ResearchState) -> str:
    if state["evidence_score"] >= EVIDENCE_THRESHOLD:
        return "answer"
    if assess(state["documents"], state["claims"]).unresolved:
        return "review"
    if state.get("retrieval_round", 1) < MAX_ROUNDS:
        return "search_again"
    return "review"


def human_review(state: ResearchState) -> dict:
    trigger = review_trigger(state)
    reply = interrupt({"trigger": trigger, "draft": limitation_text(state), "options": list(DECISIONS)})
    decision = reply.get("decision") if isinstance(reply, dict) else reply
    if decision not in DECISIONS:
        raise ValueError(f"review decision must be one of {DECISIONS}, got {decision!r}")
    note = reply.get("note", "") if isinstance(reply, dict) else ""
    return {
        "review_decision": decision,
        "history": [f"human_review: {trigger}; reviewer decided {decision}" + (f" ({note})" if note else "")],
    }


def route_after_review(state: ResearchState) -> str:
    return "publish" if state["review_decision"] == "approve" else "withhold"


def publish_reviewed(state: ResearchState) -> dict:
    return {"draft_answer": f"Approved after human review. {limitation_text(state)}",
            "history": ["publish_reviewed: limited answer published"]}


def withhold_answer(state: ResearchState) -> dict:
    return {"draft_answer": f"Withheld after human review: the reviewer did not approve an answer, because {review_trigger(state)}.",
            "history": ["withhold_answer: no answer published"]}

Compared with NB 04, the contract gains `review_decision`, written by `human_review` with the value a
person supplies, and the two new answer nodes become owners of `draft_answer` in place of
`write_limitation`.

In [ ]:
CONTRACT = {
    **LOOP_CONTRACT,
    "review_decision": {"human_review"},
    "draft_answer": {"write_answer", "publish_reviewed", "withhold_answer"},
}

builder = StateGraph(ResearchState)
builder.add_node("collect_primary", collect_primary)
builder.add_node("collect_secondary", collect_secondary)
builder.add_node("score", make_score(llm))
builder.add_node("refine_query", make_refine_query(llm))
builder.add_node("write_answer", write_answer)
builder.add_node("human_review", human_review)
builder.add_node("publish_reviewed", publish_reviewed)
builder.add_node("withhold_answer", withhold_answer)
builder.add_edge(START, "collect_primary")
builder.add_edge(START, "collect_secondary")
builder.add_edge(["collect_primary", "collect_secondary"], "score")
builder.add_conditional_edges("score", route_with_review,
                              {"answer": "write_answer", "search_again": "refine_query", "review": "human_review"})
builder.add_edge("refine_query", "collect_primary")
builder.add_edge("refine_query", "collect_secondary")
builder.add_conditional_edges("human_review", route_after_review, {"publish": "publish_reviewed", "withhold": "withhold_answer"})
builder.add_edge("write_answer", END)
builder.add_edge("publish_reviewed", END)
builder.add_edge("withhold_answer", END)

graph = builder.compile(checkpointer=InMemorySaver())
print(graph.get_graph().draw_mermaid())

### 3 Trigger 1: topic c pauses after its last round

The run searches three times, as in NB 04, and then stops at `human_review` instead of writing an
answer. What comes back is not a final state but the interruption, with what the reviewer needs.

In [ ]:
paused = graph.invoke({"question": QUESTIONS["c"]}, thread("c-approve"))
request = paused["__interrupt__"][0].value
print("trigger:", request["trigger"])
print("draft:  ", request["draft"][:150], "...")
print("options:", request["options"])
print("waiting at:", graph.get_state(thread("c-approve")).next)

The run has neither failed nor finished: it is waiting. Its state (three rounds of documents, every
claim, the evidence score) sits in the checkpoint under the thread `c-approve`, and nothing has been
published.

### 4 The reviewer decides

The reviewer approves on one thread. On a second thread with the same question, another reviewer
rejects, with a note. The decision goes to the paused node with `Command(resume=...)`.

In [ ]:
approved = graph.invoke(Command(resume={"decision": "approve"}), thread("c-approve"))
print(approved["draft_answer"][:200], "...\n")

graph.invoke({"question": QUESTIONS["c"]}, thread("c-reject"))
rejected = graph.invoke(Command(resume={"decision": "reject", "note": "one regulator source is not enough"}), thread("c-reject"))
print(rejected["draft_answer"])
print(rejected["history"][-2])

Two decisions, two routes. Approval publishes the limited answer, now marked as reviewed. Rejection
publishes nothing and says why. The reviewer's note stays in the history, so the decision can be
traced later.

### 5 Trigger 2: a constructed credible contradiction

The document below is **constructed for this notebook**; it is not part of the corpus. It is a
fictional news report, as credible as the existing one (authority 0.7), that states a different
revenue figure. The caller passes it in with the question; the collectors add the corpus documents
as usual. Its claim was read by `qwen3:8b` with the NB 03 prompt and recorded.

In [ ]:
print({key: value for key, value in CONSTRUCTED_REPORT.items() if key != "text"})

paused = graph.invoke({"question": QUESTIONS["a"], "documents": [CONSTRUCTED_REPORT]}, thread("a-constructed"))
print("\nround:  ", paused.get("retrieval_round", 1))
print("trigger:", paused["__interrupt__"][0].value["trigger"])

CHF 342 million from the press release and the first news report, CHF 320 million from the second
report: both sides clear the authority floor, neither is outdated, neither copies the other. The rules
have no basis to pick one, so the run goes to review in round 1. The reviewer sees the competing
figures with their sources, which is what a person needs to decide, for example by asking the company
which figure it regards as reported.

In [ ]:
resolved = graph.invoke(Command(resume={"decision": "approve", "note": "publish with the conflict stated"}), thread("a-constructed"))
print(resolved["draft_answer"][:230], "...")

### 6 The other questions do not pause

In [ ]:
for topic in "abde":
    final = graph.invoke({"question": QUESTIONS[topic]}, thread(f"plain-{topic}"))
    print(f"{topic}: {'paused' if '__interrupt__' in final else 'answered'} | {final.get('draft_answer', '')[:70]}")

### 7 A restart

Memory disappears with the process. A review can take days, so the checkpoints must outlive the
program. `SqliteSaver` writes them to a local file: no server, no account, no API key. The next cells
pause a run, close the connection and delete the graph, as if the notebook had been stopped, then
build a new graph on the same file and resume.

In [ ]:
database = CHECKPOINT_FOLDER / "checkpoints.sqlite"
connection = sqlite3.connect(database, check_same_thread=False)
durable_graph = builder.compile(checkpointer=SqliteSaver(connection))

durable_graph.invoke({"question": QUESTIONS["c"]}, thread("c-durable"))
print("paused and saved:", durable_graph.get_state(thread("c-durable")).next)

connection.close()
del durable_graph, connection
print("connection closed, graph deleted; file size:", database.stat().st_size, "bytes")

In [ ]:
connection = sqlite3.connect(database, check_same_thread=False)
restarted_graph = builder.compile(checkpointer=SqliteSaver(connection))

print("found waiting at:", restarted_graph.get_state(thread("c-durable")).next)
final = restarted_graph.invoke(Command(resume={"decision": "approve"}), thread("c-durable"))
print(final["draft_answer"][:120], "...")

The new graph knew nothing in memory, yet found the run waiting at `human_review` and finished it with
the full context of three search rounds. That is what makes a human-in-the-loop step practical: the
decision does not have to arrive while the program is still running.

### 8 What the checkpoints hold

Every step of a thread is a checkpoint. Listing them shows the run's path, from the latest step back to
the first.

In [ ]:
for snapshot in restarted_graph.get_state_history(thread("c-durable")):
    values = snapshot.values
    print(f"next {str(snapshot.next):<45} round {values.get('retrieval_round', 1)}  "
          f"documents {len(values.get('documents', []))}  score {values.get('evidence_score', '-')}")
connection.close()

Each line is a complete, saved state. For an audit this means the evidence behind a decision can be
shown as it was at the moment of the decision, not reconstructed afterwards. LangGraph can also
continue a thread from an earlier checkpoint; the exercise explores that.

The contract from NB 02 still applies, through the pause and the resume:

In [ ]:
check_contract(graph.stream({"question": QUESTIONS["c"]}, thread("contract-check")), CONTRACT)
check_contract(graph.stream(Command(resume={"decision": "reject"}), thread("contract-check")), CONTRACT)
print("The contract holds through pause and resume.")

## Common mistakes

### Mistake 1: pausing without a checkpointer

The same graph, compiled without a checkpointer. The pause seems to work: the first call returns the
interruption like before. Only the attempt to resume shows that nothing was saved.

In [ ]:
forgetful_graph = builder.compile()
first = forgetful_graph.invoke({"question": QUESTIONS["c"]})
print("first call returned an interruption:", "__interrupt__" in first)
forgetful_graph.invoke(Command(resume={"decision": "approve"}))

*"Cannot use Command(resume=...) without checkpointer"*. The three search rounds and the reviewer's
request are gone; the run has to start again. In a real review process, the mistake would show up
only when the reviewer replies, possibly days later. **Fix:** compile every graph that can pause with
a checkpointer, and for anything longer than a session, a durable one such as `SqliteSaver`.

### Mistake 2: resuming on the wrong thread

The run paused on the thread `c-approve` has finished; suppose a reviewer now resumes a paused run with
a typo in the thread name.

In [ ]:
graph.invoke({"question": QUESTIONS["c"]}, thread("c-review"))
print("state under the typo:", graph.get_state(thread("c-reveiw")).values)
graph.invoke(Command(resume={"decision": "approve"}), thread("c-reveiw"))

`KeyError: 'question'`. Under the misspelt name there is no paused run, so LangGraph starts the graph
from the beginning with an empty state, and the first node fails because there is no question. The
message says nothing about threads; the reviewer's decision is simply lost. **Fix:** keep the thread
name with the review task instead of retyping it, and check `get_state(...).next` before resuming: a
waiting run reports `('human_review',)`.

### Mistake 3: a decision that is not one

The reviewer replies "yes". It sounds like approval, and that is exactly why it must not be taken as
one.

In [ ]:
graph.invoke(Command(resume={"decision": "yes"}), thread("c-review"))

`human_review` refuses anything outside `approve` and `reject`, and the run stays paused, ready for a
valid decision. Input from people is checked where it enters the graph, like any other input from
outside. **Fix:** offer the allowed options with the request (the interruption lists them) and send
exactly one of them.

## Prompt log

This notebook adds no new prompt. The only new model call reads the constructed report with the NB 03
claim prompt; recorded from `qwen3:8b`, it returned `{"figure": "CHF 320 million"}` on the first
attempt, so there was nothing to iterate. Everything new here (pausing, saving, resuming, routing the
decision) is plain LangGraph and Python.

## Summary

The assistant can now stop and ask. When the evidence stays uncertain, either because the quality gate
fails after the last round or because credible sources disagree, the run saves its state, pauses with
`interrupt()`, and shows a person the trigger and a draft. The person's decision arrives with
`Command(resume=...)` on the same thread, possibly after a restart, and approval and rejection lead to
different outcomes. Checkpoints keep every step, which makes the evidence behind each decision
inspectable afterwards.

The review graph lives in the package as `persistence.py`. The next cell checks that the notebook and
the package agree; it compares structure, contract and routing in every mode, and complete runs only
in scripted mode.

In [ ]:
from research_assistant import persistence as package_persistence

package_graph = package_persistence.build_review_graph(llm, InMemorySaver())
structure = lambda g: (set(g.get_graph().nodes), {(e.source, e.target) for e in g.get_graph().edges})
assert structure(package_graph) == structure(graph)
assert package_persistence.REVIEW_CONTRACT == CONTRACT
for name in ("c-approve", "a-constructed"):
    state = graph.get_state(thread(name)).values
    assert package_persistence.review_trigger(state) == review_trigger(state)
    assert package_persistence.route_with_review(state) == route_with_review(state)
if mode == "fake":
    package_graph.invoke({"question": QUESTIONS["c"]}, thread("check"))
    final = package_graph.invoke(Command(resume={"decision": "approve"}), thread("check"))
    assert final["draft_answer"] == graph.get_state(thread("c-approve")).values["draft_answer"]
shutil.rmtree(CHECKPOINT_FOLDER, ignore_errors=True)
print("Notebook and package agree.")

**Next: NB 07, the synthesis.** Six notebooks have built one assistant step by step. NB 07 answers the
four research questions with evidence from those notebooks, compares the chain of NB 01 with the graph
of NB 06, names the limits of the approach, and sketches the next step: several agents behind a
supervisor.

## Exercise

1. Change `route_with_review` so that a credible contradiction searches again first and only goes to
   review after the last round. Run the constructed case. How many rounds does it take, what does the
   reviewer see at the end, and why is the immediate review the better design?
2. For the thread `c-reject`, use `graph.get_state_history(...)` to find the checkpoint just after the
   scoring of round 2, and print its evidence score and documents. What would it mean for an audit to
   continue a thread from such an earlier checkpoint instead of from the latest one?